# v81 -- Cosmic Filament Alignment Along KTF³ Axis
## Galaxy pair proxies from SDSS DR12
**Author:** Andrew Cotting | 5 April 2026 | github.com/Andrew-Cot/KTF3-Analysis

Test: Are galaxy pairs (10-50 Mpc) aligned with KTF³ axis l=277°, b=-31°?
Reuses SDSS cache from v76.

In [ ]:
!pip install astroquery astropy -q
import numpy as np, matplotlib.pyplot as plt, os
from astropy.cosmology import FlatLambdaCDM
from scipy.spatial.distance import cdist

print('='*60)
print('v81 -- Filament Alignment vs KTF³ Axis')
print('KTF³: l=277°, b=-31° | SDSS DR12 galaxy pairs')
print('='*60)

cosmo = FlatLambdaCDM(H0=67.4, Om0=0.315)
l_k, b_k = np.radians(277.0), np.radians(-31.0)
axis_vec = np.array([np.cos(b_k)*np.cos(l_k),
                     np.cos(b_k)*np.sin(l_k),
                     np.sin(b_k)])


In [ ]:
from astroquery.sdss import SDSS

sdss_file = 'sdss_dr12_galaxies.npy'
if os.path.exists(sdss_file):
    data = np.load(sdss_file)
    ra, dec, z_spec = data[:,0], data[:,1], data[:,2]
    is_real = True
    print(f'Loaded {len(ra):,} cached SDSS galaxies')
else:
    print('Querying SDSS...')
    query = """
    SELECT TOP 30000 p.ra, p.dec, s.z
    FROM PhotoObj p JOIN SpecObj s ON p.objID = s.bestObjID
    WHERE s.class = 'GALAXY' AND s.z BETWEEN 0.05 AND 0.30
    AND s.zWarning = 0 AND p.ra BETWEEN 100 AND 260
    AND p.dec BETWEEN -10 AND 60 ORDER BY NEWID()
    """
    try:
        result = SDSS.query_sql(query, timeout=120)
        ra = result['ra'].data.astype(float)
        dec = result['dec'].data.astype(float)
        z_spec = result['z'].data.astype(float)
        np.save(sdss_file, np.column_stack([ra, dec, z_spec]))
        is_real = True
    except Exception as e:
        print(f'Failed: {e}')
        is_real = False
        np.random.seed(277)
        n = 15000
        ra = np.random.uniform(100,260,n)
        dec = np.random.uniform(-10,60,n)
        z_spec = np.random.uniform(0.05,0.30,n)

d_com = cosmo.comoving_distance(z_spec).value
ra_r = np.radians(ra); dec_r = np.radians(dec)
pos = np.column_stack([
    d_com*np.cos(dec_r)*np.cos(ra_r),
    d_com*np.cos(dec_r)*np.sin(ra_r),
    d_com*np.sin(dec_r)
])
print(f'N={len(pos):,}, d={d_com.min():.0f}-{d_com.max():.0f} Mpc')


In [ ]:
# Galaxy pairs at 10-50 Mpc as filament proxies
n_sample = min(2000, len(pos))
idx = np.random.choice(len(pos), n_sample, replace=False)
pos_s = pos[idx]

print('Finding pairs at 10-50 Mpc...')
dists = cdist(pos_s, pos_s)
i_arr, j_arr = np.where((dists > 10) & (dists < 50))
mask = i_arr < j_arr
i_arr, j_arr = i_arr[mask], j_arr[mask]
print(f'{len(i_arr):,} pairs found')

pair_vecs = pos_s[j_arr] - pos_s[i_arr]
norms = np.linalg.norm(pair_vecs, axis=1, keepdims=True)
pair_vecs_norm = pair_vecs / norms
alignment = np.abs(pair_vecs_norm @ axis_vec)
obs_mean = alignment.mean()

np.random.seed(277)
n_boot = 1000
boot_means = [alignment[np.random.choice(len(alignment), len(alignment))].mean()
              for _ in range(n_boot)]
boot_means = np.array(boot_means)
sigma = (obs_mean - boot_means.mean()) / boot_means.std()

ctrl_sigmas = []
for _ in range(200):
    l_r2 = np.random.uniform(0,360); b_r2 = np.degrees(np.arcsin(np.random.uniform(-1,1)))
    ax_r = np.array([np.cos(np.radians(b_r2))*np.cos(np.radians(l_r2)),
                     np.cos(np.radians(b_r2))*np.sin(np.radians(l_r2)),
                     np.sin(np.radians(b_r2))])
    al_r = np.abs(pair_vecs_norm @ ax_r)
    ctrl_sigmas.append((al_r.mean() - boot_means.mean()) / boot_means.std())
ctrl_sigmas = np.array(ctrl_sigmas)
p_val = np.mean(np.abs(ctrl_sigmas) >= np.abs(sigma))

print(f'\nRESULT:')
print(f'  obs mean |cos|: {obs_mean:.6f}')
print(f'  bootstrap mean: {boot_means.mean():.6f}')
print(f'  SIGMA: {sigma:+.3f}')
print(f'  p-value: {p_val:.3f}')
verdict = "SIGNAL" if abs(sigma)>2 else ("HINT" if abs(sigma)>1 else "NULL")
print(f'  VERDICT: {verdict}')

fig, axes = plt.subplots(1,3,figsize=(18,6))
fig.suptitle(f'v81 -- Filament Alignment | σ={sigma:+.2f} | {"SDSS DR12" if is_real else "SYNTHETIC"}',
             fontsize=13, fontweight='bold')

axes[0].hist(alignment, bins=30, color='#3498db', alpha=0.7, edgecolor='black', density=True)
axes[0].plot(np.linspace(0,1,100), 2*np.ones(100), 'r--', lw=2, label='Isotropic')
axes[0].axvline(obs_mean, color='green', lw=2.5, label=f'Obs={obs_mean:.4f}')
axes[0].set_xlabel('|cos(angle)| with KTF³ axis'); axes[0].legend(); axes[0].grid(True,alpha=0.3)

axes[1].hist(boot_means, bins=30, color='#95a5a6', alpha=0.8, edgecolor='black')
axes[1].axvline(obs_mean, color='green', lw=2.5, label=f'σ={sigma:+.2f}')
axes[1].set_xlabel('Bootstrap mean'); axes[1].legend(); axes[1].grid(True,alpha=0.3)

axes[2].hist(ctrl_sigmas, bins=25, color='#95a5a6', alpha=0.8, edgecolor='black', density=True)
axes[2].axvline(sigma, color='red', lw=2.5, label=f'KTF³ σ={sigma:+.2f}')
axes[2].set_title(f'200 random axes | p={p_val:.3f} [{verdict}]')
axes[2].legend(); axes[2].grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('v81_boss_filament.png', dpi=150, bbox_inches='tight')
plt.show()
print('GitHub: github.com/Andrew-Cot/KTF3-Analysis | OSF: osf.io/42nks')
